# TwinStock AI — Granite Time Series Experimentation Notebook

This notebook is for **experimentation and development**. It does NOT need to run for the ML service API to work.

## Stages covered here
- STAGE 2: Load historical demand dataset
- STAGE 3: Inspect and preprocess data
- STAGE 4: Baseline moving-average forecast
- STAGE 5: IBM Granite TTM integration (requires `ibm-granite-tsfm`)
- STAGE 6: Compare baseline vs Granite (MAE, RMSE, MAPE)

---

**IMPORTANT:** The current `data/warehouse_raw.csv` contains an inventory snapshot, NOT historical demand.
To run Stages 2–6, you need a CSV with columns: `date, item_id, demand`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ML service modules (run from ml-service/ directory)
import sys
sys.path.insert(0, str(Path('.').resolve()))

from src.preprocessing import prepare_time_series, prepare_item_series
from src.forecast import (
    BaselineMovingAverageModel,
    GraniteForecastModel,
    get_model_status,
    forecast_demand,
)
from src.evaluation import (
    calculate_mae, calculate_rmse, calculate_mape,
    chronological_train_test_split, evaluate_forecast,
)

print('Imports OK')

## STAGE 2 — Load Historical Dataset

Supply a CSV with columns: `date, item_id, demand`

Example:
```
date,item_id,demand
2026-01-01,ITM10025,12
2026-01-02,ITM10025,15
```

In [ ]:
# -----------------------------------------------------------------------
# Option A: Load from a historical demand CSV
# -----------------------------------------------------------------------
# HISTORICAL_CSV = 'data/historical_demand.csv'
# df_raw = pd.read_csv(HISTORICAL_CSV)

# -----------------------------------------------------------------------
# Option B: Synthetic demo data (use this if you don't have a CSV yet)
# -----------------------------------------------------------------------
import datetime

def make_synthetic_demand(item_id: str, n_days: int = 90, base: float = 20.0, noise: float = 5.0):
    start = datetime.date(2026, 1, 1)
    dates = [start + datetime.timedelta(days=i) for i in range(n_days)]
    rng = np.random.default_rng(seed=42)
    demand = base + rng.normal(0, noise, n_days)
    demand = np.clip(demand, 0, None)  # no negative demand
    return pd.DataFrame({'date': [str(d) for d in dates], 'item_id': item_id, 'demand': demand.round(1)})

df_raw = pd.concat([
    make_synthetic_demand('ITM10025', n_days=90, base=15.0),
    make_synthetic_demand('ITM10026', n_days=90, base=30.0),
], ignore_index=True)

print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

## STAGE 3 — Inspect & Preprocess

In [ ]:
df = prepare_time_series(df_raw)
print(f'After preprocessing: {df.shape}')
print(f'Items: {df["item_id"].unique().tolist()}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
df.head()

In [ ]:
# Select one item
ITEM_ID = 'ITM10025'

item_df = prepare_item_series(df, ITEM_ID)
print(f'Item: {ITEM_ID}')
print(f'Data points: {len(item_df)}')
item_df.head()

## STAGE 4 — Baseline Forecast (Moving Average)

This is a DEVELOPMENT BASELINE only. Not AI, not Granite.

In [ ]:
history_list = [
    {'date': str(row['date'].date() if hasattr(row['date'], 'date') else row['date']),
     'demand': float(row['demand'])}
    for _, row in item_df.iterrows()
]

train_history, test_history = chronological_train_test_split(history_list, test_ratio=0.3)
print(f'Train: {len(train_history)} points | Test: {len(test_history)} points')

In [ ]:
baseline_model = BaselineMovingAverageModel(window=7)

baseline_forecast = baseline_model.predict(train_history, horizon=len(test_history))

print(f'Model: {baseline_model.model_name}')
print(f'Forecast horizon: {len(baseline_forecast)} days')
baseline_forecast[:5]

In [ ]:
metrics = evaluate_forecast(test_history, baseline_forecast)
print('--- Baseline Evaluation ---')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
train_dates  = [pd.to_datetime(p['date']) for p in train_history]
train_vals   = [p['demand'] for p in train_history]
test_dates   = [pd.to_datetime(p['date']) for p in test_history]
test_vals    = [p['demand'] for p in test_history]
pred_vals    = [p['predicted_demand'] for p in baseline_forecast]

plt.figure(figsize=(12, 4))
plt.plot(train_dates, train_vals, label='Training demand', color='steelblue')
plt.plot(test_dates, test_vals, label='Actual demand (test)', color='green')
plt.plot(test_dates, pred_vals, label=f'Forecast ({baseline_model.model_name})', color='orange', linestyle='--')
plt.title(f'Demand Forecast — {ITEM_ID}  |  MAE={metrics["mae"]}  RMSE={metrics["rmse"]}  MAPE={metrics["mape"]}%')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.tight_layout()
plt.show()

## STAGE 5 — IBM Granite Time Series (Granite TTM)

**This stage requires:**
```bash
pip install ibm-granite-tsfm torch transformers
```

Once installed, implement `GraniteForecastModel.load()` in `src/forecast.py`.

The integration point:
```python
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
from tsfm_public import TimeSeriesForecastingPipeline

self._model = TinyTimeMixerForPrediction.from_pretrained('ibm/TTM')
self._pipeline = TimeSeriesForecastingPipeline(
    model=self._model,
    timestamp_column='date',
    target_columns=['demand'],
    freq='D',
)
```

In [ ]:
# Check current model status
status = get_model_status()
print('Model status:')
for k, v in status.items():
    print(f'  {k}: {v}')

## STAGE 6 — Compare Baseline vs Granite

Run this section after Granite is loaded (STAGE 5 complete).

In [ ]:
from src.forecast import get_active_model

active_model = get_active_model()
print(f'Active model: {active_model.model_name}')
print(f'Available: {active_model.is_available}')

if active_model.is_available:
    granite_forecast = active_model.predict(train_history, horizon=len(test_history))
    granite_metrics = evaluate_forecast(test_history, granite_forecast)
    
    print('\n--- Baseline ---')
    for k, v in metrics.items():
        print(f'  {k}: {v}')
    
    print(f'\n--- {active_model.model_name} ---')
    for k, v in granite_metrics.items():
        print(f'  {k}: {v}')
else:
    print('Granite not available yet — complete STAGE 5 first.')

---

## Notes

- This notebook is **standalone** — the ML service API runs without it.
- Never fake Granite predictions. If it is not available, say so.
- The baseline model is labelled `"Baseline (Moving Average)"` in all responses.
- After STAGE 5, restart the ML service to activate Granite for live requests.